In [1]:
!pip install pymupdf
!pip install chromadb sentence-transformers
!pip install rank_bm25


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access the API key from .env
api_key = os.getenv('GEMINI_API_KEY')

# Note: genai.configure() will be called in the cell where genai is imported

In [3]:
# CELL 2: Imports & Token Estimator
import fitz  # PyMuPDF
import json

def estimate_tokens(text):
    """Rough approximation: 1 token is usually about 4 characters."""
    return len(text) // 4

In [4]:
# CELL 3: Parsing, Cleaning & Chunking Logic (FIXED TEXT EXTRACTION)
import fitz

def extract_and_chunk_pdf(file_path, doc_name, source_url):
    doc = fitz.open(file_path)
    chunks = []
    
    current_section = "General Guidance"
    current_chunk_text = ""
    chunk_index = 1
    
    TARGET_TOKENS = 250
    OVERLAP_TOKENS = 40

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        page_height = page.rect.height
        blocks = page.get_text("dict")["blocks"]
        
        for block in blocks:
            if "lines" not in block:
                continue
                
            # Filter header/footer noise
            _, y0, _, y1 = block["bbox"]
            if y1 < (page_height * 0.07) or y0 > (page_height * 0.93):
                continue
                
            block_text = ""
            for line in block["lines"]:
                line_spans = [s["text"] for s in line["spans"] if s["text"].strip()]
                if line_spans:
                    block_text += " ".join(line_spans) + " "
            
            block_text = block_text.replace("- ", "").strip()
            if not block_text or len(block_text) < 15:
                continue
            
            # Check if this block looks like a major section heading
            first_span = block["lines"][0]["spans"][0]
            if (first_span["size"] > 12 or "Bold" in first_span["font"]) and len(block_text) < 80:
                current_section = block_text
                continue
            
            current_chunk_text += block_text + " "
            
            # Commit chunk once target size is reached
            if estimate_tokens(current_chunk_text) >= TARGET_TOKENS:
                chunks.append({
                    "document_name": doc_name,
                    "page_number": page_num + 1,
                    "section_title": current_section,
                    "chunk_id": f"{doc_name.lower().replace(' ', '_').replace('/', '_')}_p{page_num+1}_c{chunk_index:03d}",
                    "source_url": source_url,
                    "text": current_chunk_text.strip()
                })
                words = current_chunk_text.split()
                overlap_words = words[-OVERLAP_TOKENS:] if len(words) > OVERLAP_TOKENS else []
                current_chunk_text = " ".join(overlap_words) + " "
                chunk_index += 1

    # Flush leftover text
    if estimate_tokens(current_chunk_text) > 40:
        chunks.append({
            "document_name": doc_name,
            "page_number": len(doc),
            "section_title": current_section,
            "chunk_id": f"{doc_name.lower().replace(' ', '_').replace('/', '_')}_p{len(doc)}_c{chunk_index:03d}",
            "source_url": source_url,
            "text": current_chunk_text.strip()
        })
        
    return chunks

In [5]:
# CELL 4: Execution & Quality Check
# Ensure "NICE_NG226.pdf" is uploaded to your current working directory
pdf_filename = "NICE_NG226.pdf" 

print("Parsing PDF and generating chunks...")
guideline_chunks = extract_and_chunk_pdf(
    file_path=r"C:\Users\joeel\OneDrive\Desktop\AI_Hackathon_OsteoGuard-AI\osteoguard_ai\DOCs\Osteoarthritis(NICE).pdf",
    doc_name="NICE NG226 Osteoarthritis",
    source_url="https://www.nice.org.uk/guidance/ng226"
)

print(f"Total chunks created: {len(guideline_chunks)}")

# Print a sample chunk safely to verify the metadata schema
print("\n--- Sample Chunk Output ---")
if len(guideline_chunks) > 0:
    print(json.dumps(guideline_chunks[0], indent=2))
else:
    print("No chunks generated. The PDF might be image-based (scanned) or the text was filtered out.")

Parsing PDF and generating chunks...
Total chunks created: 92

--- Sample Chunk Output ---
{
  "document_name": "NICE NG226 Osteoarthritis",
  "page_number": 2,
  "section_title": "Your responsibility",
  "chunk_id": "nice_ng226_osteoarthritis_p2_c001",
  "source_url": "https://www.nice.org.uk/guidance/ng226",
  "text": "The recommendations in this guideline represent the view of NICE, arrived at after careful  consideration of the evidence available. When exercising their judgement, professionals  and practitioners are expected to take this guideline fully into account, alongside the  individual needs, preferences and values of their patients or the people using their service.  It is not mandatory to apply the recommendations, and the guideline does not override the  responsibility to make decisions appropriate to the circumstances of the individual, in  consultation with them and their families and carers or guardian. All problems (adverse events) related to a medicine or medical dev

In [6]:
# CELL 5: Process the ACR/AF Guideline and Combine
# Update this variable to match your exact ACR/AF PDF file name
acr_pdf_filename = r"C:\Users\joeel\OneDrive\Desktop\AI_Hackathon_OsteoGuard-AI\osteoguard_ai\DOCs\Osteoarthritis(ACR).pdf"

print("Parsing ACR/AF PDF and generating chunks...")
acr_chunks = extract_and_chunk_pdf(
    file_path=acr_pdf_filename,
    doc_name="ACR/AF 2019 Osteoarthritis Guideline",
    source_url="https://rheumatology.org/osteoarthritis-guideline" 
)

print(f"ACR/AF chunks created: {len(acr_chunks)}")

# Combine both document chunk lists into one master dataset
master_knowledge_base = guideline_chunks + acr_chunks

print(f"\nTotal chunks in combined knowledge base: {len(master_knowledge_base)}")

# Print a sample from the new document to verify metadata
if acr_chunks:
    print("\n--- Sample ACR/AF Chunk Output ---")
    print(json.dumps(acr_chunks[5], indent=2))
else:
    print("No chunks generated for ACR/AF. Check the PDF filename and path.")

Parsing ACR/AF PDF and generating chunks...
ACR/AF chunks created: 67

Total chunks in combined knowledge base: 159

--- Sample ACR/AF Chunk Output ---
{
  "document_name": "ACR/AF 2019 Osteoarthritis Guideline",
  "page_number": 2,
  "section_title": "ACR/AF GUIDELINE FOR MANAGEMENT OF HAND, HIP, AND KNEE OA  | \u2009\u2009\u2009\u2002\u2003\u2002221",
  "chunk_id": "acr_af_2019_osteoarthritis_guideline_p2_c006",
  "source_url": "https://rheumatology.org/osteoarthritis-guideline",
  "text": "and to develop the recommendations (6). ACR policy guided management of conflicts of interest and \u00addisclosures (https\u200b://www.rheum\u200batolo\u200bgy.org/Pract\u200bice-Quali\u200bty/Clini\u200bcal Suppo\u200brt/Clini\u200bcal-Pract\u200bice-Guide\u200blines/\u200bOsteo\u200barthr\u200bitis). A full de\u00adscription of the methods is presented in Supplementary Appendix 1 (on the Arthritis & Rheumatology web site at http://onlin\u200belibr\u200bary. wiley.com/doi/10.1002/art.41142/\u200b

In [7]:
# CELL 6: Embedding, Indexing, and Baseline Test (OPTIMIZED)
import chromadb
from chromadb.utils import embedding_functions

print("Initializing Vector Database with Medical Embeddings...")
chroma_client = chromadb.PersistentClient(path="./osteoarthritis_db")

# NEW: Use a clinical-specific embedding model
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="NeuML/pubmedbert-base-embeddings")

# NEW: Use a v2 collection to avoid dimension mismatch with the old database
collection = chroma_client.get_or_create_collection(
    name="osteoarthritis_guidelines_v2",
    embedding_function=sentence_transformer_ef
)

documents = [chunk["text"] for chunk in master_knowledge_base]
metadatas = [{
    "document_name": chunk["document_name"],
    "page_number": chunk["page_number"],
    "section_title": chunk["section_title"],
    "source_url": chunk["source_url"]
} for chunk in master_knowledge_base]
ids = [chunk["chunk_id"] for chunk in master_knowledge_base]

print(f"Embedding and indexing {len(documents)} chunks. This may take a minute...")
# Use upsert instead of add to safely overwrite if rerun
collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
print("Indexing complete!\n")

test_query = "What is the recommended initial physical therapy or exercise for knee osteoarthritis?"
print(f"QUERY: {test_query}\n")
results = collection.query(query_texts=[test_query], n_results=3)

for i in range(len(results['documents'][0])):
    print(f"--- Result {i+1} ---")
    print(f"Text: {results['documents'][0][i][:200]}...\n")

Initializing Vector Database with Medical Embeddings...


c:\Users\joeel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4565.58it/s]


Embedding and indexing 159 chunks. This may take a minute...
Indexing complete!

QUERY: What is the recommended initial physical therapy or exercise for knee osteoarthritis?

--- Result 1 ---
Text: training with and without props such as elastic bands, and isometric exercise. Neuromuscular training has been developed to address muscle weakness, reduced sensorimotor control, and functional instab...

--- Result 2 ---
Text: balance or reducing weight bearing on the affected joint. The guideline committee has made the following recommendations for research. What is the clinical and cost effectiveness of supervised group a...

--- Result 3 ---
Text: and instruction in use of and fitting of splints and braces in their practices. Most patients with OA are likely to experience benefit from referral to physical therapy and/or occupational therapy at ...



In [8]:
# CELL 7: Retrieval Evaluation & Precision Testing
# Load your persistent database
import chromadb
chroma_client = chromadb.PersistentClient(path=r"C:\Users\joeel\OneDrive\Desktop\AI_Hackathon_OsteoGuard-AI\osteoguard_ai\DB\osteoarthritis_db")
collection = chroma_client.get_collection("osteoarthritis_clinical_guidelines")

# Step 4: Mini Evaluation Set
# Expand this list to 15-20 questions for a full evaluation
eval_queries = [
    "When should antihypertensive treatment be started?", # Out-of-scope test[cite: 2]
    "What is the recommended initial physical therapy or exercise for knee osteoarthritis?",
    "Are topical NSAIDs recommended before oral NSAIDs?",
    "When is joint replacement surgery considered for the hip?",
    "Should patients with hand osteoarthritis use orthoses?"
]

# Step 1: Tune Top-K[cite: 2]
K = 5 

print(f"--- RUNNING EVALUATION (Top-{K}) ---\n")

for query in eval_queries:
    print(f"QUERY: {query}")
    results = collection.query(
        query_texts=[query],
        n_results=K
    )
    
    # Step 6: Design the evidence panel[cite: 2]
    for i in range(len(results['documents'][0])):
        score = results['distances'][0][i] if 'distances' in results and results['distances'] else "N/A"
        chunk_id = results['ids'][0][i]
        doc = results['metadatas'][0][i]['document_name']
        page = results['metadatas'][0][i]['page_number']
        section = results['metadatas'][0][i]['section_title']
        
        print(f"  Chunk {i+1} • {doc} • Section: {section} • Page: {page}")
        # Print a short preview of the text for manual validation
        print(f"  Preview: {results['documents'][0][i][:150]}...\n")
    print("-" * 60)

--- RUNNING EVALUATION (Top-5) ---

QUERY: When should antihypertensive treatment be started?


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5127.82it/s]


  Chunk 1 • ACR/AF 2019 Osteoarthritis Guideline • Section: DISCUSSION • Page: 12
  Preview: or (anti-­NGF) agents suggest that significant analgesic benefits may occur but that incompletely explained important safety issues may arise. A small...

  Chunk 2 • ACR/AF 2019 Osteoarthritis Guideline • Section: Comprehensive management of OA • Page: 4
  Preview: A comprehensive plan for the management of OA in an individual patient may include educational, behavioral, psycho- social, and physical interventions...

  Chunk 3 • ACR/AF 2019 Osteoarthritis Guideline • Section: METHODS • Page: 3
  Preview: tical Industries (less than $10,000 each). No other disclosures relevant to this article were reported. Address correspondence to Sharon L. Kolasinski...

  Chunk 4 • ACR/AF 2019 Osteoarthritis Guideline • Section: Pharmacologic management (Table 2) • Page: 10
  Preview: e mainstay of the pharmacologic man- agement of OA, and their use is strongly recommended. A large number of trials have est

In [9]:
# CELL 8: Ultimate Search (Aggressive Filtering & Context Injection)
import numpy as np
import re
from rank_bm25 import BM25Okapi
import chromadb
from sentence_transformers import CrossEncoder

# 1. Load ChromaDB
chroma_client = chromadb.PersistentClient(path="./osteoarthritis_db")
collection = chroma_client.get_collection("osteoarthritis_guidelines_v2")

# 2. Build Keyword (BM25) Index
all_data = collection.get(include=["documents", "metadatas"])
all_docs = all_data["documents"]
all_ids = all_data["ids"]
all_metadatas = all_data["metadatas"]

def tokenize_clean(text):
    return re.sub(r'\W+', ' ', text).lower().split()

tokenized_corpus = [tokenize_clean(doc) for doc in all_docs]
bm25 = BM25Okapi(tokenized_corpus)

# 3. Load Cross-Encoder Model
print("Loading Cross-Encoder model...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def reciprocal_rank_fusion(semantic_ranks, keyword_ranks, k=60):
    rrf_scores = {}
    for rank, chunk_id in enumerate(semantic_ranks):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank + 1)
    for rank, chunk_id in enumerate(keyword_ranks):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)

# 4. The Advanced Reranked Search
def reranked_hybrid_search(query, top_n=5, fetch_k=40):
    
    # HACK 1: Expanded junk list to kill "Research" and "Rationale" sections
    JUNK_SECTIONS = [
        "Contents", "REFERENCES", "ACKNOWLEDGMENTS", "METHODS", "Overview",
        "Recommendations for research", "Key recommendations for research",
        "Other recommendations for research", "Rationale and impact", "Context",
        "Update information", "Your responsibility"
    ]
    
    semantic_results = collection.query(query_texts=[query], n_results=fetch_k)
    semantic_ranked_ids = semantic_results["ids"][0]
    
    keyword_scores = bm25.get_scores(tokenize_clean(query))
    top_keyword_indices = np.argsort(keyword_scores)[::-1][:fetch_k]
    keyword_ranked_ids = [all_ids[i] for i in top_keyword_indices]
    
    fused_results = reciprocal_rank_fusion(semantic_ranked_ids, keyword_ranked_ids)[:fetch_k]
    
    cross_inp = []
    candidate_chunks = []
    
    for chunk_id, _ in fused_results:
        original_idx = all_ids.index(chunk_id)
        metadata = all_metadatas[original_idx]
        text = all_docs[original_idx]
        
        # Apply the junk filter
        if any(junk.lower() in metadata['section_title'].lower() for junk in JUNK_SECTIONS):
            continue
            
        # HACK 2: Context Injection for the Cross-Encoder
        enriched_text = f"Section: {metadata['section_title']}. {text}"
        
        cross_inp.append([query, enriched_text])
        candidate_chunks.append((chunk_id, metadata, text))
        
        if len(cross_inp) == 20: 
            break
        
    cross_scores = cross_encoder.predict(cross_inp)
    reranked_pairs = sorted(zip(cross_scores, candidate_chunks), key=lambda x: x[0], reverse=True)
    
    return reranked_pairs[:top_n]

Loading Cross-Encoder model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5611.72it/s]


In [12]:
# CELL 9: Query Definitions and Ground Truth Mappings
eval_queries = [
    # Pharmacological
    "What is the clinical role of duloxetine in managing knee osteoarthritis?",
    "Are topical NSAIDs strongly recommended for hand osteoarthritis?",
    "What is the guideline stance on oral NSAIDs vs topical NSAIDs?",
    "Is acetaminophen/paracetamol recommended as a first-line treatment for OA?",
    "What does the guideline say about tramadol for OA pain?",
    "Are non-tramadol opioids recommended for osteoarthritis management?",
    "Should capsaicin be used for knee osteoarthritis?",
    "Is glucosamine recommended for modifying disease progression in OA?",
    "What is the recommendation for chondroitin sulfate in hand OA?",
    "Is there clinical evidence supporting Vitamin D supplementation for knee OA?",
    "Should fish oil supplements be prescribed for OA?",
    "What are the cardiovascular risks associated with oral NSAID use in OA?",
    "Are gastroprotective agents recommended when prescribing oral NSAIDs?",
    "Is topical lidocaine effective for hip osteoarthritis?",
    "What is the role of bisphosphonates in treating osteoarthritis?",
    
    # Intra-articular / Injections
    "Are intra-articular corticosteroid injections recommended for knee OA?",
    "How does the guideline view hyaluronic acid injections for hip OA?",
    "Is platelet-rich plasma (PRP) recommended for knee osteoarthritis?",
    "What is the stance on stem cell injections for joint cartilage repair?",
    "Are prolotherapy injections supported by evidence for OA?",
    "Is botulinum toxin recommended for treating osteoarthritis pain?",
    "How often can intra-articular corticosteroids be safely administered?",
    
    # Non-Pharmacological / Physical
    "What non-pharmacological interventions are universally recommended for osteoarthritis?",
    "How much weight loss is recommended to see clinical benefits in overweight OA patients?",
    "What type of exercise is recommended for knee osteoarthritis?",
    "Is aquatic exercise better than land-based exercise for hip OA?",
    "Does the guideline recommend Tai Chi for osteoarthritis?",
    "Is yoga recommended for managing knee osteoarthritis symptoms?",
    "What is the clinical recommendation regarding acupuncture for OA?",
    "Is transcutaneous electrical nerve stimulation (TENS) recommended for knee OA?",
    "What is the evidence for therapeutic ultrasound in OA management?",
    "Are thermal interventions (heat/cold) recommended for symptom relief?",
    "Is massage therapy a strongly recommended intervention for OA?",
    "What does the guideline recommend regarding manual therapy with exercise?",
    "Is pulsed electromagnetic field therapy recommended for OA?",
    
    # Devices / Orthotics / Surgical
    "What does the guideline recommend for the use of canes or walking sticks?",
    "Are tibiofemoral knee braces recommended for knee OA?",
    "Should patients with hand OA use thumb carpometacarpal (CMC) orthoses?",
    "Are laterally wedged insoles recommended for medial compartment knee OA?",
    "What is the stance on shock-absorbing footwear for hip OA?",
    "Is kinesiotaping recommended for osteoarthritis?",
    "Are walking frames or rollators recommended for end-stage hip OA?",
    "When should a patient be referred for joint replacement surgery?",
    "Is arthroscopic lavage and debridement recommended for knee OA?",
    "What are the indications for osteotomy in knee osteoarthritis?",
    
    # Complex / Edge Cases
    "How should comorbid gastrointestinal issues affect NSAID prescription?",
    "What interventions are conditionally recommended against for hand OA?",
    "Are there different exercise recommendations for hand OA vs knee OA?",
    "How should chronic kidney disease impact pharmacological choices for OA?",
    "What is the sequence of escalating treatments for a patient failing topical NSAIDs?"
]

# Ground Truth: Substrings that must appear in the retrieved text for it to be a relevant hit
ground_truth_keywords = {
    "What is the clinical role of duloxetine in managing knee osteoarthritis?": ["duloxetine"],
    "Are topical NSAIDs strongly recommended for hand osteoarthritis?": ["topical nsaid", "hand oa"],
    "What is the guideline stance on oral NSAIDs vs topical NSAIDs?": ["oral nsaid", "topical nsaid"],
    "Is acetaminophen/paracetamol recommended as a first-line treatment for OA?": ["acetaminophen", "paracetamol"],
    "What does the guideline say about tramadol for OA pain?": ["tramadol"],
    "Are non-tramadol opioids recommended for osteoarthritis management?": ["non-tramadol", "opioid"],
    "Should capsaicin be used for knee osteoarthritis?": ["capsaicin", "knee"],
    "Is glucosamine recommended for modifying disease progression in OA?": ["glucosamine"],
    "What is the recommendation for chondroitin sulfate in hand OA?": ["chondroitin", "hand"],
    "Is there clinical evidence supporting Vitamin D supplementation for knee OA?": ["vitamin d"],
    "Should fish oil supplements be prescribed for OA?": ["fish oil"],
    "What are the cardiovascular risks associated with oral NSAID use in OA?": ["cardiovascular", "nsaid"],
    "Are gastroprotective agents recommended when prescribing oral NSAIDs?": ["gastroprotect", "nsaid"],
    "Is topical lidocaine effective for hip osteoarthritis?": ["lidocaine"],
    "What is the role of bisphosphonates in treating osteoarthritis?": ["bisphosphonate"],
    "Are intra-articular corticosteroid injections recommended for knee OA?": ["corticosteroid", "glucocorticoid"],
    "How does the guideline view hyaluronic acid injections for hip OA?": ["hyaluronic", "hyaluronan", "hip"],
    "Is platelet-rich plasma (PRP) recommended for knee osteoarthritis?": ["platelet-rich", "prp"],
    "What is the stance on stem cell injections for joint cartilage repair?": ["stem cell"],
    "Are prolotherapy injections supported by evidence for OA?": ["prolotherapy"],
    "Is botulinum toxin recommended for treating osteoarthritis pain?": ["botulinum"],
    "How often can intra-articular corticosteroids be safely administered?": ["corticosteroid", "short-term", "weeks"],
    "What non-pharmacological interventions are universally recommended for osteoarthritis?": ["exercise", "weight loss", "non-pharmacological"],
    "How much weight loss is recommended to see clinical benefits in overweight OA patients?": ["weight loss", "5%", "10%"],
    "What type of exercise is recommended for knee osteoarthritis?": ["exercise", "walking", "strengthening"],
    "Is aquatic exercise better than land-based exercise for hip OA?": ["aquatic"],
    "Does the guideline recommend Tai Chi for osteoarthritis?": ["tai chi"],
    "Is yoga recommended for managing knee osteoarthritis symptoms?": ["yoga"],
    "What is the clinical recommendation regarding acupuncture for OA?": ["acupuncture"],
    "Is transcutaneous electrical nerve stimulation (TENS) recommended for knee OA?": ["transcutaneous", "tens"],
    "What is the evidence for therapeutic ultrasound in OA management?": ["ultrasound"],
    "Are thermal interventions (heat/cold) recommended for symptom relief?": ["thermal", "heat", "cold"],
    "Is massage therapy a strongly recommended intervention for OA?": ["massage"],
    "What does the guideline recommend regarding manual therapy with exercise?": ["manual therapy"],
    "Is pulsed electromagnetic field therapy recommended for OA?": ["pulsed", "electromagnetic", "vibration"],
    "What does the guideline recommend for the use of canes or walking sticks?": ["cane", "walking stick", "walking aid"],
    "Are tibiofemoral knee braces recommended for knee OA?": ["tibiofemoral", "brace"],
    "Should patients with hand OA use thumb carpometacarpal (CMC) orthoses?": ["cmc", "orthos"],
    "Are laterally wedged insoles recommended for medial compartment knee OA?": ["wedged insole"],
    "What is the stance on shock-absorbing footwear for hip OA?": ["footwear", "shoe"],
    "Is kinesiotaping recommended for osteoarthritis?": ["kinesiotaping", "tape"],
    "Are walking frames or rollators recommended for end-stage hip OA?": ["walking aid", "assistive device"],
    "When should a patient be referred for joint replacement surgery?": ["joint replacement", "referral"],
    "Is arthroscopic lavage and debridement recommended for knee OA?": ["arthroscopic", "lavage", "debridement"],
    "What are the indications for osteotomy in knee osteoarthritis?": ["osteotomy", "surgery"],
    "How should comorbid gastrointestinal issues affect NSAID prescription?": ["gastrointestinal", "gastroprotection", "nsaid"],
    "What interventions are conditionally recommended against for hand OA?": ["hand oa", "conditionally recommended against"],
    "Are there different exercise recommendations for hand OA vs knee OA?": ["hand oa", "knee oa", "exercise"],
    "How should chronic kidney disease impact pharmacological choices for OA?": ["kidney", "renal", "nsaid"],
    "What is the sequence of escalating treatments for a patient failing topical NSAIDs?": ["topical nsaid", "oral nsaid"]
}

In [13]:
# CELL 10: Batch Evaluator with Metrics
import numpy as np

K = 5

total_batch_precision = 0.0
total_batch_confidence = 0.0

print(f"--- EXPANDED BATCH EVALUATION WITH METRICS (Top-{K}) ---\n")

for query_idx, query in enumerate(eval_queries, 1):
    print("=" * 60)
    print(f"QUERY [{query_idx}/{len(eval_queries)}]: {query}")
    print("=" * 60)
    
    results = reranked_hybrid_search(query, top_n=K)
    
    relevant_hits = 0
    query_confidences = []
    keywords = ground_truth_keywords.get(query, [])
    
    for rank, (score, (chunk_id, metadata, text)) in enumerate(results):
        # 1. Convert logit score to 0-100% confidence via Sigmoid
        confidence_pct = (1.0 / (1.0 + np.exp(-score))) * 100.0
        query_confidences.append(confidence_pct)
        
        # 2. Check relevance against required keywords
        is_relevant = any(kw.lower() in text.lower() for kw in keywords) if keywords else True
        if is_relevant:
            relevant_hits += 1
            
        print(f"[Rank {rank+1} | Score: {score:.2f} | Confidence: {confidence_pct:.1f}% | Relevant: {is_relevant}]")
        print(f"Source: {metadata.get('document_name', 'N/A')} | Section: {metadata.get('section_title', 'N/A')}")
        print(f"Text snippet: {text[:200]}...\n")
    
    # Compute query-level metrics
    p_at_k = (relevant_hits / K) * 100.0
    avg_query_confidence = np.mean(query_confidences)
    
    total_batch_precision += p_at_k
    total_batch_confidence += avg_query_confidence
    
    print(f">> Query Precision@{K}: {p_at_k:.1f}%")
    print(f">> Query Avg Confidence: {avg_query_confidence:.1f}%\n")

# Compute dataset-level summary metrics
final_precision = total_batch_precision / len(eval_queries)
final_confidence = total_batch_confidence / len(eval_queries)

print("=" * 60)
print("FINAL EVALUATION METRICS")
print("=" * 60)
print(f"Total Evaluated Queries: {len(eval_queries)}")
print(f"Mean Precision@{K}:        {final_precision:.2f}%")
print(f"Mean Model Confidence:   {final_confidence:.2f}%")
print("=" * 60)

--- EXPANDED BATCH EVALUATION WITH METRICS (Top-5) ---

QUERY [1/50]: What is the clinical role of duloxetine in managing knee osteoarthritis?
[Rank 1 | Score: 3.66 | Confidence: 97.5% | Relevant: True]
Source: NICE NG226 Osteoarthritis | Section: 10 Topical and oral medicines
Text snippet: Full details of the evidence and the committee's discussion are in evidence review G: clinical and cost effectiveness of electrotherapy for the management of osteoarthritis . What is the clinical and ...

[Rank 2 | Score: 2.63 | Confidence: 93.3% | Relevant: True]
Source: ACR/AF 2019 Osteoarthritis Guideline | Section: ACR/AF GUIDELINE FOR MANAGEMENT OF HAND, HIP, AND KNEE OA  |       229
Text snippet: future investigations specific to OA. Evidence suggests that duloxetine has efficacy in the treatment of OA when used alone or in combination with NSAIDs; however, there are issues regarding tolerabil...

[Rank 3 | Score: 1.69 | Confidence: 84.5% | Relevant: True]
Source: ACR/AF 2019 Osteoarthritis Gu